In [3]:
import os, json, sqlite3
from datetime import date, datetime
from dotenv import load_dotenv
from openai import OpenAI
from PIL import Image, ImageDraw, ImageFont
import gradio

In [4]:
load_dotenv(override=True)
MODEL = "gpt-4.1-mini"
openai = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

### system prompt

In [5]:
HOSPITAL_NAME = "Lanka Care Hospital"
DB_NAME = "hospital.db"

system_message = f"""
You are a friendly, caring virtual assistant for {HOSPITAL_NAME}.
Reply in the same language the patient uses (English, Sinhala or Singlish). Keep answers short, warm and polite, 2 to 3 sentences.
Today's date is {date.today()} ({date.today().strftime('%A')}).

You can: register new patients, show doctor details, channel (book) a doctor, cancel a channeling, and explain a patient's test report.

STRICT RULES:
- NEVER recommend, suggest, name or give doses of any medicine or treatment to anyone. If asked, politely say you cannot advise on medicine and that they should consult a doctor, and offer to channel one.
- NEVER diagnose. For test reports only read out the stored results and notes; tell the patient to discuss them with their doctor.
- Never invent doctors, dates, prices or results. Use the tools; if unsure, say so.
- If the patient describes serious symptoms (chest pain, difficulty breathing, heavy bleeding, etc.), tell them to go to the emergency unit immediately or call 1990.

Flows:
- New patient: ask name, phone, age, gender. Then call register_patient. Tell them their Patient ID and where the card is saved.
- Channel: ask Patient ID (or offer to register), which doctor or specialty, and the date (YYYY-MM-DD). Use get_doctors to show options first. Then call book_appointment, and tell them the appointment ID, appointment number, fee and receipt location.
- Cancel: ask appointment ID and the phone number, then call cancel_appointment.
- Test report: ask Patient ID and registered phone number, then call get_test_reports.
"""


### DATABASE

In [6]:
def init_db():
    conn = sqlite3.connect(DB_NAME)
    conn.executescript("""
    CREATE TABLE IF NOT EXISTS patients (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        name TEXT, phone TEXT, age INTEGER, gender TEXT, registered_on TEXT
    );
    CREATE TABLE IF NOT EXISTS doctors (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        name TEXT, specialty TEXT, qualifications TEXT,
        available_days TEXT, time_slot TEXT, room TEXT, fee INTEGER
    );
    CREATE TABLE IF NOT EXISTS appointments (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        patient_id INTEGER, doctor_id INTEGER, appt_date TEXT,
        appt_number INTEGER, fee INTEGER, status TEXT
    );
    CREATE TABLE IF NOT EXISTS reports (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        patient_id INTEGER, test_name TEXT, test_date TEXT,
        result TEXT, notes TEXT
    );
    """)
    # dummy data (only once)
    if conn.execute("SELECT COUNT(*) FROM doctors").fetchone()[0] == 0:
        conn.executemany(
            "INSERT INTO doctors (name, specialty, qualifications, available_days, time_slot, room, fee) VALUES (?,?,?,?,?,?,?)",
            [
                ("Dr. Sunil Fernando", "Cardiologist", "MBBS, MD, MRCP", "Monday,Wednesday,Friday", "4:00 PM - 7:00 PM", "Room 101", 3500),
                ("Dr. Anjali Perera", "Pediatrician", "MBBS, DCH, MD (Paeds)", "Tuesday,Thursday,Saturday", "9:00 AM - 12:00 PM", "Room 205", 3000),
                ("Dr. Ruwan Silva", "General Physician", "MBBS, MD", "Monday,Tuesday,Wednesday,Thursday,Friday", "5:00 PM - 8:00 PM", "Room 110", 2000),
                ("Dr. Nadeesha Jayasinghe", "Dermatologist", "MBBS, MD (Derm)", "Wednesday,Saturday", "2:00 PM - 5:00 PM", "Room 302", 3200),
                ("Dr. Kamal Wijesinghe", "Orthopedic Surgeon", "MBBS, MS (Ortho), FRCS", "Monday,Thursday", "3:00 PM - 6:00 PM", "Room 208", 4000),
            ])
        conn.execute("INSERT INTO patients (name, phone, age, gender, registered_on) VALUES (?,?,?,?,?)",
                     ("Nimal Perera", "0771234567", 45, "Male", str(date.today())))
        conn.executemany(
            "INSERT INTO reports (patient_id, test_name, test_date, result, notes) VALUES (?,?,?,?,?)",
            [
                (1, "Full Blood Count (FBC)", "2026-10-20",
                 "Hemoglobin 13.8 g/dL, WBC 6,500 /uL, Platelets 250,000 /uL", "All values within normal range."),
                (1, "Fasting Blood Sugar", "2026-10-20",
                 "108 mg/dL", "Slightly above normal range (70-100 mg/dL). Please discuss with your doctor."),
                (1, "Lipid Profile", "2026-10-21",
                 "Total Cholesterol 215 mg/dL, LDL 140 mg/dL, HDL 45 mg/dL", "Borderline high. Please consult your doctor."),
            ])
    conn.commit()
    conn.close()

init_db()

def pid(n):  # patient code like PT0001
    return f"PT{int(n):04d}"

def parse_pid(p):
    return int(str(p).upper().replace("PT", "").strip())
